<a href="https://colab.research.google.com/github/unni527/Detection-of-Parkinson-s-Disease-Integrating-Speech-based-Chromagram-Features-and-Vision-Transformer/blob/main/Parkinsons_ViT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import timm

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ----------------------------
# 1. Define Lightweight Vision Transformer
# ----------------------------
class PretrainedViT_Tiny(nn.Module):
    def __init__(self, model_name="vit_tiny_patch16_224", num_classes=1):
        super().__init__()
        self.base_model = timm.create_model(model_name, pretrained=True)
        in_features = self.base_model.head.in_features
        self.base_model.head = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.base_model(x)

# ----------------------------
# 2. Data Loading
# ----------------------------
transform = transforms.Compose([
    transforms.Resize((224,224)),   # match pretrained ViT input size
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])
'''
transform = transforms.Compose([
    transforms.Resize((128,128)),   # smaller resolution -> faster training
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])
'''
# root folder containing two subfolders: Healthy, Diseased
dataset = datasets.ImageFolder("/content/drive/MyDrive/Chroma", transform=transform)

# Get labels for StratifiedKFold
labels = np.array(dataset.targets)

# ----------------------------
# 3. 5-Fold Cross-Validation
# ----------------------------
skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    print(f"\n===== Fold {fold+1} =====")

    train_subset = Subset(dataset, train_idx)
    val_subset   = Subset(dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=10, shuffle=True)
    val_loader   = DataLoader(val_subset, batch_size=10, shuffle=False)

    # Model, Loss, Optimizer
    model = PretrainedViT_Tiny().to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-5)

    train_acc, val_acc, train_loss, val_loss = [], [], [], []

    # ---------------- Training ----------------
    for epoch in range(10):  # exactly 13 epochs per fold
        model.train()
        correct, total, epoch_loss = 0, 0, 0
        for imgs, labels_batch in train_loader:
            imgs, labels_batch = imgs.to(device), labels_batch.to(device).float().unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels_batch)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            preds = (outputs > 0.5).int()
            correct += (preds == labels_batch.int()).sum().item()
            total += labels_batch.size(0)

        train_acc.append(100 * correct / total)
        train_loss.append(epoch_loss / len(train_loader))

        # Validation
        model.eval()
        correct, total, epoch_loss = 0, 0, 0
        with torch.no_grad():
            for imgs, labels_batch in val_loader:
                imgs, labels_batch = imgs.to(device), labels_batch.to(device).float().unsqueeze(1)
                outputs = model(imgs)
                loss = criterion(outputs, labels_batch)

                epoch_loss += loss.item()
                preds = (outputs > 0.5).int()
                correct += (preds == labels_batch.int()).sum().item()
                total += labels_batch.size(0)

        val_acc.append(100 * correct / total)
        val_loss.append(epoch_loss / len(val_loader))

        print(f"Epoch [{epoch+1}/13], Train Acc: {train_acc[-1]:.4f}%, Val Acc: {val_acc[-1]:.4f}%")

    # ---------------- Evaluation ----------------
    y_true, y_pred, y_scores = [], [], []
    model.eval()
    with torch.no_grad():
        for imgs, labels_batch in val_loader:
            imgs = imgs.to(device)
            outputs = model(imgs).cpu().numpy().ravel()
            preds = (outputs > 0.5).astype(int)
            y_scores.extend(outputs)
            y_pred.extend(preds)
            y_true.extend(labels_batch.numpy())

    # Classification Report
    print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=dataset.classes))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset.classes, yticklabels=dataset.classes)
    plt.title(f"Confusion Matrix (Fold {fold+1})")
    plt.show()

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
    plt.plot([0,1],[0,1],'--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve (Fold {fold+1})")
    plt.legend()
    plt.show()

    # Accuracy curves
    plt.plot(train_acc, label="Train Acc")
    plt.plot(val_acc, label="Val Acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title(f"Training and Validation Accuracy (Fold {fold+1})")
    plt.legend()
    plt.show()

    # Loss curves
    plt.plot(train_loss, label="Train Loss")
    plt.plot(val_loss, label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Training and Validation Loss (Fold {fold+1})")
    plt.legend()
    plt.show()

    # Save fold results
    fold_results.append({
        "fold": fold+1,
        "val_acc": val_acc[-1],
        "roc_auc": roc_auc
    })

# ----------------------------
# 4. Final Summary
# ----------------------------
print("\n===== Cross-Validation Summary =====")
for r in fold_results:
    print(f"Fold {r['fold']}: Val Acc = {r['val_acc']:.4f}%, AUC = {r['roc_auc']:.4f}")

mean_acc = np.mean([r["val_acc"] for r in fold_results])
mean_auc = np.mean([r["roc_auc"] for r in fold_results])
print(f"\nAverage Val Accuracy: {mean_acc:.4f}%")
print(f"Average AUC: {mean_auc:.4f}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import timm

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ----------------------------
# 1. Define Vision Transformer
# ----------------------------
class PretrainedViT(nn.Module):
    def __init__(self, model_name="vit_base_patch16_224", num_classes=1):
        super().__init__()
        self.base_model = timm.create_model(model_name, pretrained=True)
        in_features = self.base_model.head.in_features
        self.base_model.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
            nn.Sigmoid()  # binary output
        )
    def forward(self, x):
        return self.base_model(x)

# ----------------------------
# 2. Data Loading
# ----------------------------
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

# root folder containing two subfolders: Healthy, Diseased
dataset = datasets.ImageFolder("/content/drive/MyDrive/Chroma", transform=transform)

# Get labels for StratifiedKFold
labels = np.array([dataset.targets[i] for i in range(len(dataset))])

# ----------------------------
# 3. 5-Fold Cross-Validation
# ----------------------------
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    print(f"\n===== Fold {fold+1} =====")

    # Subset the dataset for this fold
    train_subset = Subset(dataset, train_idx)
    val_subset   = Subset(dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=100, shuffle=True)
    val_loader   = DataLoader(val_subset, batch_size=100, shuffle=False)

    # Model, Loss, Optimizer
    model = PretrainedViT().to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    train_acc, val_acc, train_loss, val_loss = [], [], [], []

    # ---------------- Training ----------------
    for epoch in range(50):  # adjust epochs as needed
        model.train()
        correct, total, epoch_loss = 0, 0, 0
        for imgs, labels_batch in train_loader:
            imgs, labels_batch = imgs.to(device), labels_batch.to(device).float().unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels_batch)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            preds = (outputs > 0.5).int()
            correct += (preds == labels_batch.int()).sum().item()
            total += labels_batch.size(0)

        train_acc.append(100 * correct / total)
        train_loss.append(epoch_loss / len(train_loader))

        # Validation
        model.eval()
        correct, total, epoch_loss = 0, 0, 0
        with torch.no_grad():
            for imgs, labels_batch in val_loader:
                imgs, labels_batch = imgs.to(device), labels_batch.to(device).float().unsqueeze(1)
                outputs = model(imgs)
                loss = criterion(outputs, labels_batch)

                epoch_loss += loss.item()
                preds = (outputs > 0.5).int()
                correct += (preds == labels_batch.int()).sum().item()
                total += labels_batch.size(0)

        val_acc.append(100 * correct / total)
        val_loss.append(epoch_loss / len(val_loader))

        print(f"Epoch [{epoch+1}/10], Train Acc: {train_acc[-1]:.4f}%, Val Acc: {val_acc[-1]:.4f}%")

    # ---------------- Evaluation ----------------
    y_true, y_pred, y_scores = [], [], []
    model.eval()
    with torch.no_grad():
        for imgs, labels_batch in val_loader:
            imgs = imgs.to(device)
            outputs = model(imgs).cpu().numpy().ravel()
            preds = (outputs > 0.5).astype(int)
            y_scores.extend(outputs)
            y_pred.extend(preds)
            y_true.extend(labels_batch.numpy())

    # Classification Report
    print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=dataset.classes, digits=4))
    print(f'Classification Report for Best Epoch (Fold {fold + 1}):\n')
    # print(classification_report(best_y_true, best_y_pred, target_names=["HC", "PD"], digits=4))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset.classes, yticklabels=dataset.classes)
    plt.title(f"Confusion Matrix (Fold {fold+1})")
    plt.show()

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
    plt.plot([0,1],[0,1],'--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve (Fold {fold+1})")
    plt.legend()
    plt.show()

    # Accuracy curves
    plt.plot(train_acc, label="Train Acc")
    plt.plot(val_acc, label="Val Acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title(f"Training and Validation Accuracy (Fold {fold+1})")
    plt.legend()
    plt.show()

    # Loss curves
    plt.plot(train_loss, label="Train Loss")
    plt.plot(val_loss, label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Training and Validation Loss (Fold {fold+1})")
    plt.legend()
    plt.show()

    # Save fold results
    fold_results.append({
        "fold": fold+1,
        "val_acc": val_acc[-1],
        "roc_auc": roc_auc
    })

# ----------------------------
# 4. Final Summary
# ----------------------------
print("\n===== Cross-Validation Summary =====")
for r in fold_results:
    print(f"Fold {r['fold']}: Val Acc = {r['val_acc']:.4f}%, AUC = {r['roc_auc']:.4f}")

mean_acc = np.mean([r["val_acc"] for r in fold_results])
mean_auc = np.mean([r["roc_auc"] for r in fold_results])
print(f"\nAverage Val Accuracy: {mean_acc:.4f}%")
print(f"Average AUC: {mean_auc:.4f}")


MEL

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import timm

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ----------------------------
# 1. Define Vision Transformer
# ----------------------------
class PretrainedViT(nn.Module):
    def __init__(self, model_name="vit_base_patch16_224", num_classes=1):
        super().__init__()
        self.base_model = timm.create_model(model_name, pretrained=True)
        in_features = self.base_model.head.in_features
        self.base_model.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
            nn.Sigmoid()  # binary output
        )
    def forward(self, x):
        return self.base_model(x)

# ----------------------------
# 2. Data Loading
# ----------------------------
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

# root folder containing two subfolders: Healthy, Diseased
dataset = datasets.ImageFolder("/content/drive/MyDrive/PD", transform=transform)

# Get labels for StratifiedKFold
labels = np.array([dataset.targets[i] for i in range(len(dataset))])

# ----------------------------
# 3. 5-Fold Cross-Validation
# ----------------------------
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
    print(f"\n===== Fold {fold+1} =====")

    # Subset the dataset for this fold
    train_subset = Subset(dataset, train_idx)
    val_subset   = Subset(dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
    val_loader   = DataLoader(val_subset, batch_size=64, shuffle=False)

    # Model, Loss, Optimizer
    model = PretrainedViT().to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    train_acc, val_acc, train_loss, val_loss = [], [], [], []

    # ---------------- Training ----------------
    for epoch in range(13):  # adjust epochs as needed
        model.train()
        correct, total, epoch_loss = 0, 0, 0
        for imgs, labels_batch in train_loader:
            imgs, labels_batch = imgs.to(device), labels_batch.to(device).float().unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels_batch)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            preds = (outputs > 0.5).int()
            correct += (preds == labels_batch.int()).sum().item()
            total += labels_batch.size(0)

        train_acc.append(100 * correct / total)
        train_loss.append(epoch_loss / len(train_loader))

        # Validation
        model.eval()
        correct, total, epoch_loss = 0, 0, 0
        with torch.no_grad():
            for imgs, labels_batch in val_loader:
                imgs, labels_batch = imgs.to(device), labels_batch.to(device).float().unsqueeze(1)
                outputs = model(imgs)
                loss = criterion(outputs, labels_batch)

                epoch_loss += loss.item()
                preds = (outputs > 0.5).int()
                correct += (preds == labels_batch.int()).sum().item()
                total += labels_batch.size(0)

        val_acc.append(100 * correct / total)
        val_loss.append(epoch_loss / len(val_loader))

        print(f"Epoch [{epoch+1}/10], Train Acc: {train_acc[-1]:.4f}%, Val Acc: {val_acc[-1]:.4f}%")

    # ---------------- Evaluation ----------------
    y_true, y_pred, y_scores = [], [], []
    model.eval()
    with torch.no_grad():
        for imgs, labels_batch in val_loader:
            imgs = imgs.to(device)
            outputs = model(imgs).cpu().numpy().ravel()
            preds = (outputs > 0.5).astype(int)
            y_scores.extend(outputs)
            y_pred.extend(preds)
            y_true.extend(labels_batch.numpy())

    # Classification Report
    print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=dataset.classes, digits=4))
    print(f'Classification Report for Best Epoch (Fold {fold + 1}):\n')
    # print(classification_report(best_y_true, best_y_pred, target_names=["HC", "PD"], digits=4))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=dataset.classes, yticklabels=dataset.classes)
    plt.title(f"Confusion Matrix (Fold {fold+1})")
    plt.show()

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
    plt.plot([0,1],[0,1],'--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve (Fold {fold+1})")
    plt.legend()
    plt.show()

    # Accuracy curves
    plt.plot(train_acc, label="Train Acc")
    plt.plot(val_acc, label="Val Acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title(f"Training and Validation Accuracy (Fold {fold+1})")
    plt.legend()
    plt.show()

    # Loss curves
    plt.plot(train_loss, label="Train Loss")
    plt.plot(val_loss, label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Training and Validation Loss (Fold {fold+1})")
    plt.legend()
    plt.show()

    # Save fold results
    fold_results.append({
        "fold": fold+1,
        "val_acc": val_acc[-1],
        "roc_auc": roc_auc
    })

# ----------------------------
# 4. Final Summary
# ----------------------------
print("\n===== Cross-Validation Summary =====")
for r in fold_results:
    print(f"Fold {r['fold']}: Val Acc = {r['val_acc']:.4f}%, AUC = {r['roc_auc']:.4f}")

mean_acc = np.mean([r["val_acc"] for r in fold_results])
mean_auc = np.mean([r["roc_auc"] for r in fold_results])
print(f"\nAverage Val Accuracy: {mean_acc:.4f}%")
print(f"Average AUC: {mean_auc:.4f}")
